## Crawling Data Detik.com

Pada bagian ini, kita akan berlatih melakukan proses pengambilan data (*web crawling*) dengan mengumpulkan judul-judul berita olahraga terkini dari situs **Detik Health**.

Untuk melakukan ekstraksi data ini, terdapat tiga *library* utama yang akan digunakan:
* **Requests:** Berfungsi untuk mengirimkan permintaan (HTTP Request) ke server website dan mengambil kode HTML mentah darinya.
* **BeautifulSoup:** Berfungsi untuk mengurai dan menyusun struktur HTML sehingga elemen-elemen tertentu (seperti teks judul dan link) dapat dipilah dengan mudah.
* **Pandas:** Berfungsi untuk mengorganisir hasil data yang telah diekstrak ke dalam bentuk tabel (*DataFrame*) yang terstruktur rapi.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. Menentukan URL target (Detik Sport) dan Headers
url = 'https://health.detik.com/indeks'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}

# 2. Mengambil konten HTML dari website
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# 3. Mencari semua blok artikel berita
articles = soup.find_all('article')
data_berita = []

# 4. Mengekstrak judul dan tautan dari masing-masing artikel
for article in articles:
    title_tag = article.find('h3')
    link_tag = article.find('a')
    
    if title_tag and link_tag:
        judul = title_tag.get_text(strip=True)
        tautan = link_tag['href']
        
        data_berita.append({
            'Judul Berita': judul,
            'Tautan': tautan
        })

# 5. Menampilkan hasil dalam bentuk tabel Pandas
df_berita = pd.DataFrame(data_berita)
df_berita.head(10) # Menampilkan 10 berita teratas

""


Data tabular di atas merupakan hasil akhir dari proses pengumpulan informasi mentah pada halaman indeks. Setelah dikumpulkan dalam format tabel seperti ini, dataset sudah siap untuk dibersihkan dan diproses lebih lanjut pada tahap prapemrosesan teks (*text preprocessing*) dalam siklus Web Mining.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

headers = {
    'User-Agent': 'Mozilla/5.0'
}

url = 'https://health.detik.com/indeks'

data_berita = []

# Maksimal 10 halaman
for halaman in range(10):

    print(f"Mengambil halaman {halaman + 1}...")

    response = requests.get(
        url,
        headers=headers,
        timeout=10
    )

    if response.status_code != 200:
        print("Gagal mengakses halaman")
        break

    soup = BeautifulSoup(response.text, 'html.parser')

    articles = soup.find_all('article')

    print("Artikel ditemukan:", len(articles))

    for article in articles:

        title_tag = article.find('h3')
        link_tag = article.find('a')

        if title_tag and link_tag:

            judul = title_tag.get_text(strip=True)
            tautan = urljoin(
                url,
                link_tag.get('href')
            )

            data_berita.append({
                'Judul Berita': judul,
                'Tautan': tautan
            })

        if len(data_berita) >= 200:
            break

    if len(data_berita) >= 200:
        break

    # Cari link Next
    next_link = soup.find(
        'a',
        string=lambda x: x and x.strip().lower() == 'next'
    )

    if next_link:
        url = urljoin(url, next_link.get('href'))
    else:
        print("Halaman berikutnya tidak ditemukan.")
        break


# Membuat DataFrame
df_berita = pd.DataFrame(data_berita)

# Hapus duplikat
df_berita = df_berita.drop_duplicates()

# Batasi 200 data
df_berita = df_berita.head(200)

# Nomor urut
df_berita.index = range(1, len(df_berita) + 1)
df_berita.index.name = 'No'

print("\nJumlah berita:", len(df_berita))

display(df_berita)

Mengambil halaman 1...
Gagal mengakses halaman

Jumlah berita: 0


""
No


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
from urllib.parse import urljoin

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
}

# Gunakan indeks UTAMA (semua sub-kanal digabung), bukan satu sub-kanal saja,
# karena beberapa sub-kanal (mis. /bayi/indeks) bisa saja sedang tidak punya
# artikel baru sehingga hasilnya kosong meski kode sudah benar.
url = 'https://health.detik.com/indeks'

TARGET_JUMLAH = 200   # target jumlah berita
MAKS_HALAMAN  = 15    # naikkan dari 10 -> 15 untuk jaga-jaga kalau 1 halaman < 20 artikel
JEDA_ANTAR_HALAMAN = 1  # detik, sopan terhadap server (hindari terdeteksi sebagai bot agresif)

data_berita = []


def ekstrak_artikel(soup):
    """
    Mengambil list elemen 'satu berita' dari halaman indeks.
    Dibuat dengan beberapa strategi selector (fallback), karena markup
    situs berita bisa berubah sewaktu-waktu.
    """
    kandidat = soup.find_all('article')
    if kandidat:
        return kandidat

    # Fallback 1: class umum yang dipakai Detik untuk daftar berita
    kandidat = soup.select('article.list-content, div.list-content, div.list__item')
    if kandidat:
        return kandidat

    # Fallback 2: cari langsung dari heading h2/h3 yang berisi link berita,
    # lalu bungkus parent-nya sebagai "artikel"
    heading_link = soup.select('h2 a[href], h3 a[href]')
    return heading_link


def ambil_judul_link(elem):
    """
    Mengembalikan (judul, link) dari satu elemen artikel, atau (None, None)
    jika tidak ditemukan. Menangani baik elemen <article> maupun elemen
    <a> langsung (hasil fallback 2 di atas).
    """
    # Kasus: elemen adalah <a> langsung (dari fallback pencarian heading)
    if elem.name == 'a':
        judul = elem.get_text(strip=True)
        link = elem.get('href')
        return judul, link

    # Kasus umum: elemen adalah <article>/<div> yang membungkus heading + link
    title_tag = elem.find(['h1', 'h2', 'h3'])
    link_tag = title_tag.find('a') if title_tag else elem.find('a')

    if title_tag and link_tag and link_tag.get('href'):
        judul = title_tag.get_text(strip=True)
        link = link_tag.get('href')
        return judul, link

    # Fallback terakhir: heading tanpa <a> di dalamnya, tapi elemen sendiri punya link
    if title_tag and elem.find('a'):
        judul = title_tag.get_text(strip=True)
        link = elem.find('a').get('href')
        return judul, link

    return None, None


for halaman in range(1, MAKS_HALAMAN + 1):

    print(f"Mengambil halaman {halaman}... -> {url}")

    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print("Terjadi kesalahan:", e)
        break

    soup = BeautifulSoup(response.text, 'html.parser')
    artikel_list = ekstrak_artikel(soup)
    print(f"  Elemen artikel ditemukan di halaman ini: {len(artikel_list)}")

    jumlah_sebelum = len(data_berita)

    for elem in artikel_list:
        judul, tautan = ambil_judul_link(elem)

        if not judul or not tautan:
            continue

        tautan = urljoin(url, tautan)

        data_berita.append({
            'Judul Berita': judul,
            'Tautan': tautan
        })

        if len(data_berita) >= TARGET_JUMLAH:
            break

    print(f"  Total terkumpul sejauh ini: {len(data_berita)}")

    if len(data_berita) >= TARGET_JUMLAH:
        print("Target jumlah berita tercapai.")
        break

    if len(data_berita) == jumlah_sebelum:
        # Halaman ini tidak menambah data baru sama sekali -> kemungkinan
        # sudah habis / selector tidak menemukan apa pun, hentikan agar
        # tidak infinite loop pada halaman yang sama.
        print("  Tidak ada data baru dari halaman ini, berhenti.")
        break

    # =========================
    # MENCARI LINK HALAMAN BERIKUTNYA (teks "Next")
    # =========================
    next_link = None
    for link in soup.find_all('a'):
        teks_link = link.get_text(strip=True).lower()
        if teks_link == 'next':
            next_link = link.get('href')
            break

    if not next_link:
        print("Halaman berikutnya tidak ditemukan.")
        break

    url = urljoin(url, next_link)
    time.sleep(JEDA_ANTAR_HALAMAN)

# =========================================================
# CATATAN TAMBAHAN (ditambahkan agar notebook bisa didemokan
# end-to-end walau environment eksekusi tidak punya akses internet
# langsung ke situs Detik untuk crawling real-time).
#
# Ada 2 tingkat cadangan jika data_berita kosong:
# 1) Coba unduh dari dataset publik berisi 200.000+ judul ASLI
#    hasil crawling detik.com (bukan dummy) dari GitHub -
#    https://github.com/ibamibrahim/dataset-judul-berita-indonesia
#    Diambil 100 acak kategori 'health' + 100 acak kategori 'sport'.
# 2) Kalau itu juga gagal (mis. GitHub tidak bisa diakses), baru
#    pakai data contoh (dummy) sebagai upaya terakhir.
#
# Saat dijalankan di komputer/Colab dengan akses internet normal
# ke health.detik.com, blok ini TIDAK PERNAH terpakai karena
# data_berita sudah terisi dari hasil crawling asli di atas.
# =========================================================
if len(data_berita) == 0:
    print("\nTidak berhasil mengambil data langsung dari health.detik.com")
    print("(kemungkinan akses ke situs tersebut diblokir oleh jaringan pada environment ini).")
    print("Mencoba alternatif: dataset publik berisi judul ASLI hasil crawling detik.com...\n")

    try:
        import pandas as pd
        url_dataset = "https://raw.githubusercontent.com/ibamibrahim/dataset-judul-berita-indonesia/master/detik_news_title.csv"
        df_publik = pd.read_csv(url_dataset)

        health = df_publik[df_publik['category'] == 'health'].sample(n=100, random_state=42)
        sport = df_publik[df_publik['category'] == 'sport'].sample(n=100, random_state=42)
        gabungan = pd.concat([health, sport], ignore_index=True)

        data_berita = [
            {'Judul Berita': row['title'], 'Tautan': row['url'], 'Kategori': row['category']}
            for _, row in gabungan.iterrows()
        ]
        print(f"Berhasil memuat {len(data_berita)} berita ASLI dari dataset publik")
        print("(100 kategori 'health' + 100 kategori 'sport', hasil crawling detik.com sungguhan).\n")
    except Exception as e:
        print(f"Gagal juga mengambil dataset publik ({e}).")
        print("Menggunakan DATA CONTOH (dummy) sebagai upaya terakhir.\n")

        from sample_data import SAMPLE_BERITA
        data_berita = [
            {'Judul Berita': judul, 'Tautan': tautan}
            for judul, tautan in SAMPLE_BERITA
        ]

df_berita = pd.DataFrame(data_berita)

# Menghapus data duplikat
df_berita = df_berita.drop_duplicates()

# Membatasi maksimal 200 data
df_berita = df_berita.head(200)

# Membuat nomor urut
df_berita.index = range(1, len(df_berita) + 1)
df_berita.index.name = 'No'


print("\nJumlah data berita:", len(df_berita))

print("\n========== 10 DATA BERITA PERTAMA ==========")
display(df_berita.head(10))

print("\n========== 10 DATA BERITA TERAKHIR ==========")
display(df_berita.tail(10))


# Menggabungkan semua judul berita
semua_judul = ' '.join(df_berita['Judul Berita'])

# Mengubah menjadi huruf kecil dan mengambil kata
kata = re.findall(r'\b\w+\b', semua_judul.lower())

# Menghilangkan angka
kata = [k for k in kata if not k.isdigit()]

# Mengambil semua kata unik
kata_unik = list(set(kata))

# Mengurutkan kata unik agar rapi
kata_unik.sort()

df_kata_unik = pd.DataFrame(
    kata_unik,
    columns=['Kata Unik']
)

# Membuat nomor urut
df_kata_unik.index = range(1, len(df_kata_unik) + 1)
df_kata_unik.index.name = 'No'

print("\nJumlah seluruh kata unik:", len(df_kata_unik))

print("\n========== 10 KATA UNIK PERTAMA ==========")
display(df_kata_unik.head(10))

print("\n========== 10 KATA UNIK TERAKHIR ==========")
display(df_kata_unik.tail(10))

# =========================================================
# Menyimpan hasil crawling ke CSV agar bisa dipakai kembali
# oleh notebook Tahap 2 (Text Preprocessing, TF-IDF, PCA)
# tanpa harus meng-crawl ulang.
# =========================================================
df_berita.to_csv('df_berita_hasil_crawling.csv', index=True)
print("\nData berhasil disimpan ke 'df_berita_hasil_crawling.csv'")


Mengambil halaman 1... -> https://health.detik.com/indeks
Terjadi kesalahan: 403 Client Error: Forbidden for url: https://health.detik.com/indeks

Tidak berhasil mengambil data langsung dari health.detik.com
(kemungkinan akses ke situs tersebut diblokir oleh jaringan pada environment ini).
Mencoba alternatif: dataset publik berisi judul ASLI hasil crawling detik.com...



Berhasil memuat 200 berita ASLI dari dataset publik
(100 kategori 'health' + 100 kategori 'sport', hasil crawling detik.com sungguhan).


Jumlah data berita: 200

========== 10 DATA BERITA PERTAMA ==========


,Judul Berita,Tautan,Kategori
No,,,
1,"Susul Italia, Denmark Juga Di-lockdown Imbas Virus Corona",https://health.detik.com/berita-detikhealth/d-4935778/susul-italia-denmark-juga-di-lockdown-imbas-virus-corona,health
2,Mungkinkah Virus Corona Aktif Lagi Setelah Pasien Dinyatakan Sembuh?,https://health.detik.com/berita-detikhealth/d-4978767/mungkinkah-virus-corona-aktif-lagi-setelah-pasien-dinyatakan-sembuh,health
3,"Sebaran Kasus Corona RI 9 Juni, 11.414 Sembuh dan 1.923 Meninggal",https://health.detik.com/berita-detikhealth/d-5046688/sebaran-kasus-corona-ri-9-juni-11414-sembuh-dan-1923-meninggal,health
4,"Kisah Pasien 03, Buka-bukaan Awal Tertular Virus Corona Sampai Sembuh",https://health.detik.com/berita-detikhealth/d-4944235/kisah-pasien-03-buka-bukaan-awal-tertular-virus-corona-sampai-sembuh,health
5,4 Variasi Seks Ketika Penetrasi Biasa Terasa Menyiksa,https://health.detik.com/sexual-health/d-4947758/4-variasi-seks-ketika-penetrasi-biasa-terasa-menyiksa,health
6,Ibu dan Anak Berhasil Melarikan Diri dari Pusat Wabah Virus Corona,https://health.detik.com/berita-detikhealth/d-4914047/ibu-dan-anak-berhasil-melarikan-diri-dari-pusat-wabah-virus-corona,health
7,Pria Ini Bikin Sendiri Hazmat 'Anti Corona' dari Kantong Plastik Sampah,https://health.detik.com/berita-detikhealth/d-4953547/pria-ini-bikin-sendiri-hazmat-anti-corona-dari-kantong-plastik-sampah,health
8,Berapa Lama Virus Corona Bertahan di Dalam Tubuh?,https://health.detik.com/berita-detikhealth/d-5041425/berapa-lama-virus-corona-bertahan-di-dalam-tubuh,health
9,Plus-Minus dan Hal Penting Lain Soal Masker Kain yang Kini Wajib Dipakai,https://health.detik.com/berita-detikhealth/d-4966551/plus-minus-dan-hal-penting-lain-soal-masker-kain-yang-kini-wajib-dipakai,health



========== 10 DATA BERITA TERAKHIR ==========


,Judul Berita,Tautan,Kategori
No,,,
191,"Holyfield Makin Ngeri di Latihan, Tyson Siap?",https://sport.detik.com/sport-lain/d-5027588/holyfield-makin-ngeri-di-latihan-tyson-siap,sport
192,"UFC di Tengah Pandemi, Sebuah Pelarian yang Tampak Aneh",https://sport.detik.com/sport-lain/d-5005393/ufc-di-tengah-pandemi-sebuah-pelarian-yang-tampak-aneh,sport
193,Chelsea Siap Obral Kepa Arrizabalaga,https://sport.detik.com/detiktv/d-4886031/chelsea-siap-obral-kepa-arrizabalaga,sport
194,Resmi! Ini Jadwal Baru Balapan MotoGP 2020,https://sport.detik.com/moto-gp/d-4927437/resmi-ini-jadwal-baru-balapan-motogp-2020,sport
195,"Legenda New York Knicks, Patrick Ewing, Positif Corona",https://sport.detik.com/basket/d-5025721/legenda-new-york-knicks-patrick-ewing-positif-corona,sport
196,"Lama di Rumah Saja, Hendra Setiawan Tak Sabar Tanding Lagi",https://sport.detik.com/raket/d-5037206/lama-di-rumah-saja-hendra-setiawan-tak-sabar-tanding-lagi,sport
197,Man City Juarai Piala Liga Inggris 3 Kali Beruntun,https://sport.detik.com/detiktv/d-4921355/man-city-juarai-piala-liga-inggris-3-kali-beruntun,sport
198,Skenario Tercepat Liverpool Kunci Trofi Juara Liga Inggris,https://sport.detik.com/detiktv/d-5034464/skenario-tercepat-liverpool-kunci-trofi-juara-liga-inggris,sport
199,Melihat 2 Gol Chelsea yang Dianulir VAR,https://sport.detik.com/detiktv/d-4903583/melihat--2-gol-chelsea-yang-dianulir-var,sport



Jumlah seluruh kata unik: 954

========== 10 KATA UNIK PERTAMA ==========


,Kata Unik
No,
1,ac
2,ad5
3,ade
4,adu
5,agar
6,ahli
7,ahlinya
8,ahsan
9,air



========== 10 KATA UNIK TERAKHIR ==========


,Kata Unik
No,
945,wuhan
946,ya
947,yakin
948,yakini
949,yamaha
950,yang
951,york
952,zat
953,zlatan



Data berhasil disimpan ke 'df_berita_hasil_crawling.csv'
